# LIME 분석 (Local Interpretable Model-Agnostic Explanations)

SHAP이 **전체 모델** 관점에서 변수 중요도를 설명한다면,
LIME은 **개별 고객 1명**의 예측 결과를 설명합니다.

- 이탈 예측 고객: 왜 이탈로 예측됐는가?
- 재구매 예측 고객: 왜 재구매로 예측됐는가?

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'lime', '-q'])
print('lime 설치 완료')

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import lime
import lime.lime_tabular

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

BASE = Path('..').resolve()
SEED = 42

# 데이터 로드
df = pd.read_csv(
    BASE / 'data/02_interim/260506_feature_engineering/promotion_0_membership_v4.csv',
    encoding='utf-8-sig'
)

EXCLUDE = [
    'USER_KEY','product_code','payment_device','device_group',
    'gender','reg_date','end_date','age_group','reg_hour','is_repurchase',
]
EXCLUDE = [c for c in EXCLUDE if c in df.columns]

for col in df.select_dtypes(include='object').columns:
    if col not in EXCLUDE:
        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

FEATURES = [c for c in df.columns if c not in EXCLUDE]
X = df[FEATURES].fillna(0)
y = df['is_repurchase']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

spw = (y_train==0).sum() / (y_train==1).sum()
model = xgb.XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    scale_pos_weight=spw, subsample=0.8, colsample_bytree=0.8,
    eval_metric='logloss', early_stopping_rounds=50,
    random_state=SEED, n_jobs=-1, verbosity=0,
)
model.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
print('모델 학습 완료')

## 1. LIME Explainer 생성

In [ ]:
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data  = X_train.values,
    feature_names  = FEATURES,
    class_names    = ['이탈(0)', '재구매(1)'],
    mode           = 'classification',
    random_state   = SEED
)
print('LIME Explainer 생성 완료')

## 2. 이탈 예측 고객 설명

In [ ]:
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

# 이탈로 예측된 실제 이탈 고객 (가장 확신도 높은 순)
churn_idx = np.where((y_pred == 0) & (y_test.values == 0))[0]
top_churn_idx = churn_idx[np.argsort(y_proba[churn_idx, 0])[::-1][0]]

print(f'이탈 예측 고객 | 이탈 확률: {y_proba[top_churn_idx, 0]*100:.1f}%')
print(f'주요 특성:')
print(X_test.iloc[top_churn_idx][['duration_days','price','recency','completion_rate']].to_dict())

exp_churn = explainer.explain_instance(
    data_row       = X_test.iloc[top_churn_idx].values,
    predict_fn     = model.predict_proba,
    num_features   = 12,
    labels         = (0,)
)

fig = exp_churn.as_pyplot_figure(label=0)
fig.set_size_inches(10, 6)
plt.title(f'이탈 예측 고객 LIME 설명 (이탈 확률: {y_proba[top_churn_idx, 0]*100:.1f}%)')
plt.tight_layout()
plt.show()

## 3. 재구매 예측 고객 설명

In [ ]:
# 재구매로 예측된 실제 재구매 고객 (가장 확신도 높은 순)
repurchase_idx = np.where((y_pred == 1) & (y_test.values == 1))[0]
top_repurchase_idx = repurchase_idx[np.argsort(y_proba[repurchase_idx, 1])[::-1][0]]

print(f'재구매 예측 고객 | 재구매 확률: {y_proba[top_repurchase_idx, 1]*100:.1f}%')
print(f'주요 특성:')
print(X_test.iloc[top_repurchase_idx][['duration_days','price','recency','completion_rate']].to_dict())

exp_repurchase = explainer.explain_instance(
    data_row       = X_test.iloc[top_repurchase_idx].values,
    predict_fn     = model.predict_proba,
    num_features   = 12,
    labels         = (1,)
)

fig = exp_repurchase.as_pyplot_figure(label=1)
fig.set_size_inches(10, 6)
plt.title(f'재구매 예측 고객 LIME 설명 (재구매 확률: {y_proba[top_repurchase_idx, 1]*100:.1f}%)')
plt.tight_layout()
plt.show()

## 4. 세그먼트별 LIME 비교 (단기구독 vs 장기+활성)

In [ ]:
# 단기구독 고위험 유저 vs 장기+활성 저위험 유저 비교
df_test = df.iloc[X_test.index].copy()
df_test['pred_proba_churn'] = 1 - y_proba[:, 1]

# 단기구독 고위험 유저
high_risk = df_test[
    (df_test['duration_days'] < 31) & (df_test['pred_proba_churn'] > 0.6)
]

# 장기+활성 저위험 유저
low_risk = df_test[
    (df_test['duration_days'] >= 31) & (df_test['pred_proba_churn'] < 0.2)
]

print(f'고위험군(단기구독): {len(high_risk)}명  |  저위험군(장기+활성): {len(low_risk)}명')

if len(high_risk) > 0 and len(low_risk) > 0:
    hr_idx = X_test.index.get_loc(high_risk.index[0])
    lr_idx = X_test.index.get_loc(low_risk.index[0])

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    for ax, idx, title, label in [
        (axes[0], hr_idx, f'고위험군(단기구독) 이탈 확률: {df_test["pred_proba_churn"].iloc[hr_idx]*100:.1f}%', 0),
        (axes[1], lr_idx, f'저위험군(장기+활성) 이탈 확률: {df_test["pred_proba_churn"].iloc[lr_idx]*100:.1f}%', 1),
    ]:
        exp = explainer.explain_instance(
            data_row   = X_test.iloc[idx].values,
            predict_fn = model.predict_proba,
            num_features = 10,
            labels     = (label,)
        )
        vals = exp.as_list(label=label)
        features_l = [v[0] for v in vals]
        weights    = [v[1] for v in vals]
        colors = ['#e74c3c' if w > 0 else '#3498db' for w in weights]
        ax.barh(range(len(weights)), weights, color=colors)
        ax.set_yticks(range(len(features_l)))
        ax.set_yticklabels(features_l, fontsize=9)
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_title(title, fontsize=11)

    plt.suptitle('LIME: 고위험군 vs 저위험군 비교', fontsize=13)
    plt.tight_layout()
    plt.show()